# YOLOv5 Model Validation

This notebook provides a comprehensive workflow for validating a trained YOLOv5 model based on a structured evaluation process. It covers data preparation, prediction, metric computation (including confusion matrix and mAP), and error analysis.

In [1]:
# ============================================================================
# 1. SETUP AND CONFIGURATION
# ============================================================================
import os
import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix
import warnings

warnings.filterwarnings('ignore')

# --- Configuration ---
# Path to the dataset YAML file. This file should contain paths to train/val/test sets and class names.
DATASET_YAML = 'vehicle_dataset/data.yaml'

# Path to the trained model weights
MODEL_PATH = 'models/yolov8.pt'

# Directory to save validation results
RESULTS_DIR = 'validation_results/'

# Thresholds
CONF_THRESHOLD = 0.45
IOU_THRESHOLD = 0.5  # For mAP calculation and NMS

# --- Setup ---
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration:")
print(f"  Dataset YAML: {DATASET_YAML}")
print(f"  Model Path:   {MODEL_PATH}")
print(f"  Results Dir:  {RESULTS_DIR}")
print(f"  Confidence:   {CONF_THRESHOLD}")
print(f"  IoU:          {IOU_THRESHOLD}")

Configuration:
  Dataset YAML: vehicle_dataset/data.yaml
  Model Path:   models/yolov8.pt
  Results Dir:  validation_results/
  Confidence:   0.45
  IoU:          0.5


## Step 1: Prepare Your Data

This step ensures that the dataset is correctly structured and that the configuration file (`data.yaml`) points to the correct training, validation, and testing directories. The test data must be completely unseen during training to get an unbiased evaluation of the model's performance.

We will load the dataset configuration to verify paths and class names.

In [2]:
# ============================================================================
# 2. LOAD DATASET AND MODEL
# ============================================================================

# Load dataset configuration from YAML
try:
    with open(DATASET_YAML, 'r') as file:
        data_config = yaml.safe_load(file)
    
    TEST_IMAGE_DIR = data_config['test']  # Using 'val' set for validation as is common
    CLASS_NAMES = data_config['names']
    
    print("[OK] Dataset configuration loaded successfully.")
    print(f"  Test image directory: {TEST_IMAGE_DIR}")
    print(f"  Number of classes: {len(CLASS_NAMES)}")
    print(f"  Class names: {CLASS_NAMES}")

except Exception as e:
    print(f"[ERROR] Could not read dataset YAML file: {e}")
    raise

# Load the trained YOLOv5 model
try:
    model = YOLO(MODEL_PATH)
    print("\n[OK] Model loaded successfully.")
except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    raise

[OK] Dataset configuration loaded successfully.
  Test image directory: images/test
  Number of classes: 1
  Class names: ['vehicle']

[OK] Model loaded successfully.


## Step 2 & 4: Run Predictions and Compute Metrics

YOLOv5's `val()` method provides a streamlined way to run predictions on the validation set and compute standard object detection metrics like Precision, Recall, and mean Average Precision (mAP).

- **Precision**: TP / (TP + FP) — Of all the predictions made, how many were correct?
- **Recall**: TP / (TP + FN) — Of all the actual objects, how many did the model find?
- **mAP@.5**: The mean Average Precision calculated at an IoU threshold of 0.5. This is a primary metric for evaluating the accuracy of object detectors.
- **mAP@.5-.95**: The average mAP over different IoU thresholds (from 0.5 to 0.95). This rewards models that are accurate at higher IoU thresholds.

The results, including the confusion matrix, will be automatically saved by the `val` function.

In [3]:
# ============================================================================
# 3. RUN VALIDATION & COMPUTE METRICS
# ============================================================================

print("\nRunning model validation...")
print("This will compute Precision, Recall, mAP, and generate a confusion matrix.")

# The val() method runs inference and computes metrics
# It will save results to a directory like 'runs/detect/val'
metrics = model.val(
    data=DATASET_YAML,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    split='test',  # Using 'test' split for validation on unseen data
    save_json=True, # Saves predictions to a JSON file, useful for manual analysis
    project=RESULTS_DIR,
    name='yolo_validation_yolov8'
)

print("\nValidation complete.")
print("-" * 50)
print("Overall Metrics:")
print(f"  mAP@.5-.95: {metrics.box.map:.4f}")
print(f"  mAP@.5:     {metrics.box.map50:.4f}")
print(f"  Precision:  {metrics.box.p[0]:.4f}") # Precision for the first class
print(f"  Recall:     {metrics.box.r[0]:.4f}") # Recall for the first class
print("-" * 50)

# The results are saved in the latest 'yolo_validation' folder
validation_run_dir = metrics.save_dir
print(f"[OK] Validation results saved to: {validation_run_dir}")


Running model validation...
This will compute Precision, Recall, mAP, and generate a confusion matrix.
Ultralytics 8.4.26  Python-3.11.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650 Ti, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 397.4100.5 MB/s, size: 60.3 KB)
val: Scanning F:\projects\object-detection-for-traffic-using-yolo\vehicle_dataset\labels\test.cache... 150 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.6it/s 2.8s.3ss
                   all        150       1388       0.98      0.979      0.989      0.914
Speed: 2.3ms preprocess, 8.5ms inference, 0.0ms loss, 1.2ms postprocess per image
Saving F:\projects\object-detection-for-traffic-using-yolo\runs\detect\validation_results\yolo_validation_yolov8\predictions.json...
Results saved to F:\projects\

## Step 3 & 5: Build and Visualize the Confusion Matrix

The `model.val()` function automatically generates a confusion matrix, which is saved as an image file (`confusion_matrix.png`) in the validation results directory.

The confusion matrix helps visualize the performance of the classifier. Each row represents the instances in an actual class while each column represents the instances in a predicted class. It's a great tool for spotting where the model is getting "confused."

Let's display the generated confusion matrix.

In [4]:
# ============================================================================
# 4. VISUALIZE CONFUSION MATRIX
# ============================================================================

# Path to the confusion matrix image generated by YOLO
confusion_matrix_path = os.path.join(validation_run_dir, 'confusion_matrix.png')

if os.path.exists(confusion_matrix_path):
    print(f"[OK] Confusion matrix found at: {confusion_matrix_path}")
    
    # Display the confusion matrix
    img = cv2.imread(confusion_matrix_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print("[WARNING] Confusion matrix image not found. It might not have been generated if there were no detections or ground truths.")

[OK] Confusion matrix found at: F:\projects\object-detection-for-traffic-using-yolo\runs\detect\validation_results\yolo_validation_yolov8\confusion_matrix.png


<Figure size 1000x800 with 1 Axes>

## Step 5: Visual Inspection

Visual inspection is crucial for understanding the model's qualitative performance. We'll run `model.predict()` on a few sample images from the test set and display the results with bounding boxes overlaid.

This helps to:
- Verify correct detections.
- Identify false positives (incorrectly detected objects).
- Spot false negatives (missed objects).

In [5]:
# ============================================================================
# 5. VISUAL INSPECTION OF PREDICTIONS
# ============================================================================
import random

print("\nPerforming visual inspection on sample test images...")

# Get a list of test images - construct correct path using dataset config
test_image_dir_full = os.path.join(data_config['path'], TEST_IMAGE_DIR)

test_images = []
if os.path.exists(test_image_dir_full):
    for f in os.listdir(test_image_dir_full):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_images.append(os.path.join(test_image_dir_full, f))
elif os.path.exists(TEST_IMAGE_DIR):
    # Fallback to relative path if absolute doesn't exist
    for f in os.listdir(TEST_IMAGE_DIR):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            test_images.append(os.path.join(TEST_IMAGE_DIR, f))

# Select a few random images to display
num_to_show = min(5, len(test_images))
if num_to_show > 0:
    sample_images = random.sample(test_images, num_to_show)

    # Run prediction and display
    results = model.predict(sample_images, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)

    fig, axes = plt.subplots(num_to_show, 1, figsize=(16, 6 * num_to_show))
    if num_to_show == 1:
        axes = [axes]

    for i, (result, ax) in enumerate(zip(results, axes)):
        annotated_img = result.plot()  # BGR image with annotations
        annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
        
        ax.imshow(annotated_img_rgb)
        ax.set_title(f"Sample Prediction: {os.path.basename(result.path)}", fontsize=12)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "sample_predictions_inspection.png"))
    plt.show()
    
    print(f"\n[OK] Visual inspection complete. A plot has been saved to {RESULTS_DIR}")
else:
    print(f"[WARNING] No test images found in {test_image_dir_full} or {TEST_IMAGE_DIR}")


Performing visual inspection on sample test images...

0: 384x640 4 vehicles, 21.7ms
1: 384x640 7 vehicles, 21.7ms
2: 384x640 10 vehicles, 21.7ms
3: 384x640 6 vehicles, 21.7ms
4: 384x640 9 vehicles, 21.7ms
Speed: 1.8ms preprocess, 21.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


<Figure size 1600x3000 with 5 Axes>


[OK] Visual inspection complete. A plot has been saved to validation_results/


## Step 6 & 7: Analyze Errors and Overfitting

Error analysis involves looking at where the model failed. The `val()` results provide plots of Precision-Recall curves, which are excellent for this.

- **P_curve.png**: Precision-Recall curve. A good model has a curve that stays high for recall values.
- **R_curve.png**: Recall curve.
- **F1_curve.png**: F1-score curve, showing the balance between precision and recall.

**Detecting Overfitting/Underfitting:**
- **High training accuracy, low validation accuracy**: This is a classic sign of **overfitting**. The model has memorized the training data but doesn't generalize.
  - *Solutions*: Use more training data, apply data augmentation, use regularization, or simplify the model architecture.
- **Low accuracy on both training and validation sets**: This indicates **underfitting**. The model is too simple to capture the patterns in the data.
  - *Solutions*: Train for more epochs, use a more complex model, gather more representative data, or perform better feature engineering.

We will now display these curves from the validation run.

In [6]:
# ============================================================================
# 6. DISPLAY METRIC CURVES FOR ANALYSIS
# ============================================================================

curve_files = {
    "Precision-Recall Curve": "PR_curve.png",
    "F1 Score Curve": "F1_curve.png",
    "Precision Curve": "P_curve.png",
    "Recall Curve": "R_curve.png"
}

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, (title, filename) in enumerate(curve_files.items()):
    filepath = os.path.join(validation_run_dir, filename)
    
    if os.path.exists(filepath):
        img = cv2.imread(filepath)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(title, fontsize=14, fontweight='bold')
    else:
        axes[i].text(0.5, 0.5, f'{filename}\nNot Found', ha='center', va='center', fontsize=12)

    axes[i].axis('off')

plt.tight_layout()
plt.show()

<Figure size 1500x1200 with 4 Axes>

## Step 7: Error Analysis: Visualizing False Positives and False Negatives

To better understand our model's failures, we will manually parse the validation results to find and visualize specific examples of False Positives (FPs) and False Negatives (FNs).

- **False Positives (FPs)** are detections made by the model that do not correspond to any ground-truth object. They represent instances where the model "hallucinates" an object.
- **False Negatives (FNs)** are ground-truth objects that the model failed to detect.

This analysis is critical for identifying patterns in the model's mistakes. For example, does it consistently miss small objects or make incorrect detections in certain lighting conditions?

In [7]:
# ============================================================================
# 7. VISUALIZE FALSE POSITIVES AND FALSE NEGATIVES
# ============================================================================
import json
from pathlib import Path

print("Starting error analysis for False Positives and False Negatives...")

json_path = os.path.join(validation_run_dir, 'predictions.json')

if not os.path.exists(json_path):
    print("[ERROR] predictions.json not found. Please ensure validation was run with 'save_json=True'.")
else:
    print(f"[OK] Found predictions file at: {json_path}")
    
    with open(json_path, 'r') as f:
        predictions_raw = json.load(f)
    
    # Handle COCO format: predictions is a list with keys like image_id, file_name, bbox, score
    if isinstance(predictions_raw, list):
        predictions = predictions_raw
    else:
        predictions = list(predictions_raw.values()) if isinstance(predictions_raw, dict) else []
    
    print(f"[OK] Loaded {len(predictions)} predictions in COCO format")
    
    if len(predictions) == 0:
        print("[ERROR] No predictions found!")
    else:
        print(f"Sample prediction keys: {list(predictions[0].keys())}")

    # --- Helper function to draw boxes ---
    def draw_box(image, box, label, color, thickness=2):
        """Draws a bounding box on an image."""
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)
        
        (label_width, label_height), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(image, (x1, y1 - label_height - baseline), (x1 + label_width, y1), color, -1)
        cv2.putText(image, label, (x1, y1 - baseline), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        return image

    def xywh2xyxy(box):
        """Converts box format [x, y, width, height] to [x1, y1, x2, y2]."""
        x, y, w, h = box
        return [x, y, x + w, y + h]

    def calculate_iou(box1, box2):
        """Calculate IoU between two boxes in xyxy format."""
        xA = max(box1[0], box2[0])
        yA = max(box1[1], box2[1])
        xB = min(box1[2], box2[2])
        yB = min(box1[3], box2[3])
        
        inter_area = max(0, xB - xA) * max(0, yB - yA)
        
        box_a_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box_b_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        iou = inter_area / float(box_a_area + box_b_area - inter_area + 1e-6)
        return iou

    # Group predictions by image
    predictions_by_image = {}
    for pred in predictions:
        file_name = pred.get('file_name', '')
        if file_name not in predictions_by_image:
            predictions_by_image[file_name] = []
        predictions_by_image[file_name].append(pred)

    print(f"\n[OK] Grouped predictions by {len(predictions_by_image)} unique images")

    fp_images = []
    fp_stats = {'total': 0, 'low_conf': 0}
    
    FP_COLOR = (255, 0, 0)  # Red for False Positives
    FN_COLOR = (0, 0, 255)  # Blue for False Negatives
    TP_COLOR = (0, 255, 0)  # Green for True Positives

    # Process each image with predictions
    for file_name, preds in predictions_by_image.items():
        try:
            # Construct full image path using the dataset path from YAML config
            image_path = os.path.join(data_config['path'], TEST_IMAGE_DIR, file_name)
            
            if not os.path.exists(image_path):
                # Try alternate path construction
                image_path_alt = os.path.join(TEST_IMAGE_DIR, file_name)
                if not os.path.exists(image_path_alt):
                    print(f"[WARNING] Image not found at: {image_path} or {image_path_alt}")
                    continue
                else:
                    image_path = image_path_alt
            
            img = cv2.imread(image_path)
            if img is None:
                print(f"[WARNING] Could not read image: {image_path}")
                continue
                
            height, width, _ = img.shape
            has_fp = False
            low_conf_count = 0
            
            # For each prediction (detection)
            for pred in preds:
                bbox = pred.get('bbox', [])
                score = pred.get('score', 0.0)
                category_id = pred.get('category_id', -1)
                
                if not bbox or len(bbox) < 4:
                    continue
                
                det_box = xywh2xyxy(bbox)
                fp_stats['total'] += 1
                
                # Simple heuristic: if confidence is below threshold, mark as potential FP
                if score < CONF_THRESHOLD:
                    label = f"Low Conf: {score:.2f}"
                    img = draw_box(img, det_box, label, FP_COLOR)
                    has_fp = True
                    low_conf_count += 1
                    fp_stats['low_conf'] += 1
                else:
                    # High confidence detections (assumed TP)
                    label = f"TP (Conf: {score:.2f})"
                    img = draw_box(img, det_box, label, TP_COLOR)
            
            if has_fp:
                fp_images.append((img, file_name, low_conf_count))
                
        except Exception as e:
            print(f"[WARNING] Error processing image {file_name}: {e}")
            continue

    print(f"\nStatistics:")
    print(f"  Total detections: {fp_stats['total']}")
    print(f"  Low confidence detections (< {CONF_THRESHOLD}): {fp_stats['low_conf']}")
    print(f"  Images with low confidence detections: {len(fp_images)}")

    # --- Visualize a few examples ---
    def show_images(image_list, title, max_images=4):
        if not image_list:
            print(f"\nNo images to display for {title}.")
            return
        
        num_to_show = min(max_images, len(image_list))
        fig, axes = plt.subplots(num_to_show, 1, figsize=(16, 8 * num_to_show))
        if num_to_show == 1:
            axes = [axes]
            
        fig.suptitle(title, fontsize=20, fontweight='bold')
        
        # Sample random images
        sampled = random.sample(image_list, num_to_show)
        for i, item in enumerate(sampled):
            if len(item) == 3:
                img, fname, low_conf_count = item
                title_text = f"{fname} ({low_conf_count} low-conf)"
            else:
                img, fname = item
                title_text = fname
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[i].imshow(img_rgb)
            axes[i].set_title(title_text, fontsize=10)
            axes[i].axis('off')
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(os.path.join(RESULTS_DIR, f"{title.replace(' ', '_').lower()}_visualization.png"), dpi=100)
        plt.show()

    if fp_images:
        show_images(fp_images, "Low Confidence Detections (Below Threshold)")
    else:
        print("\n[INFO] All detections are above confidence threshold. No low-confidence predictions to display.")

    print("\n[OK] Error analysis visualization complete.")

Starting error analysis for False Positives and False Negatives...
[OK] Found predictions file at: F:\projects\object-detection-for-traffic-using-yolo\runs\detect\validation_results\yolo_validation_yolov8\predictions.json
[OK] Loaded 1387 predictions in COCO format
Sample prediction keys: ['image_id', 'file_name', 'category_id', 'bbox', 'score']

[OK] Grouped predictions by 150 unique images

Statistics:
  Total detections: 1387
  Low confidence detections (< 0.45): 0
  Images with low confidence detections: 0

[INFO] All detections are above confidence threshold. No low-confidence predictions to display.

[OK] Error analysis visualization complete.


## Step 8: Final Evaluation Report

This final step summarizes all the findings from the validation process.

The report should include:
1.  **Overall Metrics**: Key performance indicators like mAP, Precision, and Recall.
2.  **Confusion Matrix**: A visualization of class-wise performance.
3.  **Metric Curves**: PR, F1, and other curves to diagnose model behavior.
4.  **Sample Predictions**: Visual examples of the model's performance on test images.
5.  **Analysis**: A brief conclusion about whether the model is overfitting, underfitting, or performing well, along with suggestions for next steps.

In [8]:
# ============================================================================
# 7. GENERATE FINAL SUMMARY REPORT
# ============================================================================

summary_report = f"""
# YOLOv5 Model Validation Report

**Date:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
**Model:** {MODEL_PATH}
**Dataset:** {DATASET_YAML}

---

## 1. Overall Performance Metrics

| Metric       | Value  |
|--------------|--------|
| mAP@.5-.95   | {metrics.box.map:.4f}  |
| mAP@.5       | {metrics.box.map50:.4f}  |
| Precision    | {metrics.box.p[0]:.4f}  |
| Recall       | {metrics.box.r[0]:.4f}  |

---

## 2. Analysis & Interpretation

**Precision-Recall Balance:**
The PR curve and F1 score give insight into the trade-off between precision and recall. A high area under the PR curve indicates good performance across all confidence thresholds.

**Overfitting/Underfitting Check:**
- Compare these validation metrics with your training metrics.
- If training mAP is significantly higher than validation mAP, the model is likely **overfitting**.
- If both are low, the model may be **underfitting**.
- If they are close and reasonably high, the model is generalizing well.

**Error Analysis from Confusion Matrix:**
- The confusion matrix (visualized above) shows which classes are being confused for others.
- The 'background' class in the matrix represents false positives (detections that don't match any ground truth) and false negatives (ground truth objects that were missed).

---

## 3. Recommendations

- **If Overfitting:** Consider adding more diverse training data, increasing data augmentation, or using a smaller model.
- **If Underfitting:** Consider training for more epochs, using a larger model, or ensuring your dataset is representative.
- **If there are many False Positives:** Try increasing the confidence threshold (`conf`).
- **If there are many False Negatives:** Try lowering the confidence threshold (`conf`).

---

**Validation artifacts are saved in:** {validation_run_dir}
"""

# Save the report to a markdown file
report_path = os.path.join(RESULTS_DIR, "final_validation_reportv8.md")
with open(report_path, 'w') as f:
    f.write(summary_report)

print(f"\n[OK] Final summary report saved to: {report_path}")
print("="*80)
print("VALIDATION NOTEBOOK COMPLETE!")
print("="*80)


[OK] Final summary report saved to: validation_results/final_validation_reportv8.md
VALIDATION NOTEBOOK COMPLETE!
